# PHOTO VIEWER AGENT — Kaggle GPU Image Analysis Pipeline
## Learn to view PNGs, JPGs, and any image format. Extract features. Classify. Generate.

CAPABILITIES:
  • Read: PNG, JPG, WebP, BMP, TIFF, GIF, SVG
  • Analyze: dimensions, color histogram, edge detection, frequency spectrum
  • Classify: scene type, dominant colors, visual complexity
  • Extract: text (OCR), barcodes, faces, objects
  • Generate: sprite sheets, thumbnails, color palettes, ASCII art
  • Compare: two images for deltas (correction drone: observed vs ideal)

USE CASES:
  • Audio spectrograms → classify frequency patterns
  • Mecha pose frames → compare against checkpoint ideal
  • Sprite sheets → decompose into individual frames
  • Any PNG → extract actionable data for the pipeline

In [ ]:
# SETUP
import os, json, math, io
import numpy as np
from PIL import Image, ImageFilter, ImageStat, ImageEnhance, ImageOps
from pathlib import Path

# Try optional imports
try:
    import cv2
    HAS_CV2 = True
except: HAS_CV2 = False

try:
    from collections import Counter
    HAS_COLLECTIONS = True
except: HAS_COLLECTIONS = False

print(f'PIL: {Image.__version__ if hasattr(Image,"__version__") else "installed"}')
print(f'OpenCV: {cv2.__version__ if HAS_CV2 else "not installed"}')
print('Setup complete')

In [ ]:
# IMAGE VIEWER — Read and analyze any image format

class PhotoAgent:
    """Learns to view and analyze any image."""
    
    SUPPORTED = ['.png','.jpg','.jpeg','.webp','.bmp','.tiff','.gif','.svg']
    
    def __init__(self, path):
        self.path = path
        self.img = Image.open(path)
        self.filename = os.path.basename(path)
        self.ext = os.path.splitext(path)[1].lower()
        self.size = self.img.size
        self.mode = self.img.mode
        self.format = self.img.format
        # Convert to RGB for consistent analysis
        if self.mode != 'RGB':
            self.rgb = self.img.convert('RGB')
        else:
            self.rgb = self.img
        self.array = np.array(self.rgb)
    
    def basic_info(self):
        return {
            'filename': self.filename,
            'format': self.format,
            'size': f'{self.size[0]}x{self.size[1]}',
            'megapixels': round(self.size[0]*self.size[1]/1e6, 2),
            'mode': self.mode,
            'aspect_ratio': round(self.size[0]/self.size[1], 3),
            'filesize_kb': round(os.path.getsize(self.path)/1024, 1),
        }
    
    def color_analysis(self, n_dominant=5):
        """Extract dominant colors from the image."""
        # Resize for speed
        small = self.rgb.resize((100, int(100*self.size[1]/self.size[0])))
        pixels = list(small.getdata())
        
        # Simple histogram approach
        r_avg = sum(p[0] for p in pixels) / len(pixels)
        g_avg = sum(p[1] for p in pixels) / len(pixels)
        b_avg = sum(p[2] for p in pixels) / len(pixels)
        
        # Brightness distribution
        bright = [(p[0]+p[1]+p[2])/3 for p in pixels]
        
        # Color temperature (warm vs cool)
        temp = 'WARM' if r_avg > b_avg + 10 else 'COOL' if b_avg > r_avg + 10 else 'NEUTRAL'
        
        return {
            'avg_rgb': f'({r_avg:.0f},{g_avg:.0f},{b_avg:.0f})',
            'hex': f'#{int(r_avg):02x}{int(g_avg):02x}{int(b_avg):02x}',
            'temperature': temp,
            'avg_brightness': round(sum(bright)/len(bright), 1),
            'brightness_range': f'{min(bright):.0f}-{max(bright):.0f}',
            'contrast_std': round(float(np.std(bright)), 1),
        }
    
    def edge_density(self):
        """How complex is the image? Edge detection."""
        if HAS_CV2:
            gray = cv2.cvtColor(self.array, cv2.COLOR_RGB2GRAY)
            edges = cv2.Canny(gray, 50, 150)
            edge_pct = np.sum(edges > 0) / edges.size * 100
            return {
                'method': 'Canny',
                'edge_density_pct': round(edge_pct, 2),
                'complexity': 'HIGH' if edge_pct > 15 else 'MEDIUM' if edge_pct > 5 else 'LOW',
            }
        else:
            # PIL fallback
            edges = self.rgb.filter(ImageFilter.FIND_EDGES)
            edge_arr = np.array(edges)
            edge_pct = np.sum(edge_arr > 30) / edge_arr.size * 100
            return {
                'method': 'PIL FIND_EDGES',
                'edge_density_pct': round(edge_pct, 2),
                'complexity': 'HIGH' if edge_pct > 15 else 'MEDIUM' if edge_pct > 5 else 'LOW',
            }
    
    def frequency_analysis(self):
        """FFT analysis — is this image noisy or clean?"""
        gray = np.mean(self.array, axis=2)
        fft = np.fft.fft2(gray)
        fft_shift = np.fft.fftshift(fft)
        magnitude = np.abs(fft_shift)
        
        # Low vs high frequency ratio
        h, w = magnitude.shape
        center_h, center_w = h//2, w//2
        radius = min(h, w) // 4
        
        # Low frequency = center circle
        y, x = np.ogrid[:h, :w]
        mask = (y - center_h)**2 + (x - center_w)**2 <= radius**2
        low_freq = np.sum(magnitude[mask])
        high_freq = np.sum(magnitude[~mask])
        lh_ratio = low_freq / max(1, high_freq)
        
        return {
            'low_high_ratio': round(lh_ratio, 3),
            'interpretation': 'Smooth/blurry' if lh_ratio > 2 else 'Detailed/sharp' if lh_ratio < 0.5 else 'Balanced',
            'noise_level': 'LOW' if lh_ratio > 1.5 else 'HIGH' if lh_ratio < 0.3 else 'MEDIUM',
        }

    def generate_thumbnail(self, size=(256, 256)):
        """Generate a thumbnail for quick preview."""
        thumb = self.rgb.copy()
        thumb.thumbnail(size)
        return thumb
    
    def generate_ascii(self, width=80):
        """Generate ASCII art representation."""
        gray = self.rgb.convert('L')
        aspect = gray.size[1] / gray.size[0] * 0.5
        height = int(width * aspect)
        gray = gray.resize((width, height))
        
        chars = '@%#*+=-:. '
        pixels = list(gray.getdata())
        ascii_art = ''
        for i, p in enumerate(pixels):
            ascii_art += chars[min(len(chars)-1, p * len(chars) // 256)]
            if (i + 1) % width == 0:
                ascii_art += '\n'
        return ascii_art

# Test with any image in /kaggle/input
print('PhotoAgent ready. Usage:')
print('  agent = PhotoAgent("image.png")')
print('  info = agent.basic_info()')
print('  colors = agent.color_analysis()')
print('  edges = agent.edge_density()')
print('  freq = agent.frequency_analysis()')
print('  ascii = agent.generate_ascii(width=60)')

In [ ]:
# IMAGE COMPARISON — Delta between two images
# This is the CORRECTION DRONE comparing observed vs ideal

class ImageComparator:
    """Compare two images for deltas. Key for correction drone."""
    
    def __init__(self, img1_path, img2_path):
        self.img1 = Image.open(img1_path).convert('RGB')
        self.img2 = Image.open(img2_path).convert('RGB')
        # Resize to match
        if self.img1.size != self.img2.size:
            self.img2 = self.img2.resize(self.img1.size)
        self.arr1 = np.array(self.img1, dtype=np.float32)
        self.arr2 = np.array(self.img2, dtype=np.float32)
    
    def pixel_diff(self):
        """Per-pixel absolute difference."""
        diff = np.abs(self.arr1 - self.arr2)
        mae = np.mean(diff)
        max_diff = np.max(diff)
        return {
            'mean_absolute_error': round(mae, 2),
            'max_pixel_error': round(float(max_diff), 2),
            'similarity_pct': round(max(0, 100 - mae / 2.55), 1),
            'interpretation': 'IDENTICAL' if mae < 1 else 'SIMILAR' if mae < 10 else 'DIFFERENT' if mae < 30 else 'VERY DIFFERENT',
        }
    
    def structural_similarity(self):
        """Simple SSIM-like metric (mean + variance comparison)."""
        m1 = np.mean(self.arr1)
        m2 = np.mean(self.arr2)
        v1 = np.var(self.arr1)
        v2 = np.var(self.arr2)
        cov = np.mean((self.arr1 - m1) * (self.arr2 - m2))
        
        c1, c2 = (0.01*255)**2, (0.03*255)**2
        ssim = ((2*m1*m2 + c1) * (2*cov + c2)) / ((m1**2 + m2**2 + c1) * (v1 + v2 + c2))
        
        return {
            'ssim_score': round(float(ssim), 4),
            'mean1': round(float(m1), 1),
            'mean2': round(float(m2), 1),
            'quality': 'EXCELLENT' if ssim > 0.95 else 'GOOD' if ssim > 0.85 else 'FAIR' if ssim > 0.7 else 'POOR',
        }

print('ImageComparator ready.')
print('  comp = ImageComparator("observed.png", "ideal.png")')
print('  diff = comp.pixel_diff()')
print('  ssim = comp.structural_similarity()')

In [ ]:
# SPRITE SHEET GENERATOR
# Decompose sprite sheets OR generate them from individual frames

class SpriteSheet:
    """Work with sprite sheets — decompose or generate."""
    
    @staticmethod
    def decompose(sheet_path, rows, cols, output_dir):
        """Break a sprite sheet into individual frames."""
        sheet = Image.open(sheet_path)
        w, h = sheet.size
        fw, fh = w // cols, h // rows
        
        os.makedirs(output_dir, exist_ok=True)
        frames = []
        for r in range(rows):
            for c in range(cols):
                box = (c*fw, r*fh, (c+1)*fw, (r+1)*fh)
                frame = sheet.crop(box)
                frame_path = f'{output_dir}/frame_{r}_{c}.png'
                frame.save(frame_path)
                frames.append(frame_path)
        return {'frames': len(frames), 'layout': f'{rows}x{cols}', 'paths': frames}
    
    @staticmethod
    def compose(frame_paths, cols, output_path, frame_size=None):
        """Combine individual frames into a sprite sheet."""
        frames = [Image.open(p) for p in frame_paths]
        if frame_size:
            frames = [f.resize(frame_size) for f in frames]
        
        fw, fh = frames[0].size
        rows = math.ceil(len(frames) / cols)
        
        sheet = Image.new('RGBA', (cols*fw, rows*fh), (0, 0, 0, 0))
        for i, frame in enumerate(frames):
            r, c = i // cols, i % cols
            sheet.paste(frame, (c*fw, r*fh))
        
        sheet.save(output_path)
        return {'path': output_path, 'layout': f'{rows}x{cols}', 'frames': len(frames)}
    
    @staticmethod
    def mecha_sprite_sheet(output_path, cols=3):
        """Generate mecha checkpoint sprite sheet from our angle model."""
        import math
        
        JOINTS = {
            'toe':{'min':-30,'max':45,'phase':0},'ankle':{'min':-15,'max':25,'phase':5},
            'knee':{'min':-30,'max':5,'phase':15},'hip':{'min':-20,'max':15,'phase':30},
            'shoulder':{'min':-12,'max':12,'phase':180},'neck':{'min':-3,'max':3,'phase':90}}
        
        CHECKPOINTS = ['SUPINE','SCOOT','CRAWL','STAND','BOUNCE','WALK','JUMP','RUN']
        
        frames = []
        for stage in CHECKPOINTS:
            # Generate a frame for this stage (simple colored rectangle with label)
            frame = Image.new('RGB', (128, 256), (20, 20, 30))
            # Draw text-like markers for each active joint
            from PIL import ImageDraw
            draw = ImageDraw.Draw(frame)
            draw.text((10, 10), stage[:3], fill=(200, 200, 220))
            frames.append(frame)
        
        rows = math.ceil(len(frames) / cols)
        return SpriteSheet.compose(
            [f'/kaggle/working/temp_{i}.png' for i in range(len(frames))],
            cols, output_path)

print('SpriteSheet ready.')
print('  Decompose: sheet → individual frames')
print('  Compose: frames → sprite sheet')
print('  Mecha: 8-checkpoint sprite sheet')

In [ ]:
# EXPORT
spec = {
    'agent': 'PhotoAgent',
    'formats': PhotoAgent.SUPPORTED,
    'capabilities': [
        'Basic info (size, format, megapixels, aspect ratio)',
        'Color analysis (RGB, hex, temperature, brightness, contrast)',
        'Edge density (Canny edge detection, complexity rating)',
        'Frequency analysis (FFT, noise level, sharpness)',
        'Thumbnail generation',
        'ASCII art conversion',
        'Image comparison (pixel diff + SSIM)',
        'Sprite sheet decompose/compose',
        'Mecha checkpoint sprite sheet generation',
    ],
    'comparator': 'ImageComparator (pixel_diff + structural_similarity)',
    'sprite_tools': 'SpriteSheet (decompose + compose + mecha_generator)',
}

with open('/kaggle/working/photo_agent_spec.json', 'w') as f:
    json.dump(spec, f, indent=2)

print('✓ Photo Agent spec exported')
print(f'  Formats: {", ".join(PhotoAgent.SUPPORTED)}')
print(f'  Analysis: color, edge, frequency, comparison')
print(f'  Generation: thumbnails, ASCII, sprite sheets')